# 🧠 Agent 실습 3 – LangGraph 심화


### 학습 목표
1. **컨텍스트 관리** – 대화가 길어질 때 무엇을 LLM에 전달할 것인가 (Trimming / 요약)
2. **메모리** – 대화 내용을 세션 간에 유지하는 방법 (Checkpointer)
3. **구조화된 응답** – LLM 출력을 코드에서 안정적으로 다루는 방법 (Structured Output)
4. **Human-in-the-Loop** – 특정 시점에서 사람이 개입하는 방법 (`interrupt`)


---
## ⚙️ 1. 환경 설정

In [ ]:
!pip install openai langgraph langchain langchain-openai pydantic python-dotenv -q

In [ ]:
from typing import Annotated, Literal
from typing_extensions import TypedDict
from dotenv import load_dotenv

from langchain_openai import ChatOpenAI
from langchain_core.messages import (
    HumanMessage, AIMessage, SystemMessage, trim_messages
)
from langchain_core.tools import tool
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode
from langgraph.checkpoint.memory import MemorySaver
from langgraph.types import interrupt, Command
from pydantic import BaseModel, Field

load_dotenv()
MODEL = "gpt-4o-mini"
print("✅ 환경 설정 완료")

---
## 🗜️ 2. 컨텍스트 관리 (Context Management)

### 문제: 대화가 길어지면 무슨 일이 생기나

LLM은 한 번에 처리할 수 있는 토큰 수에 한계가 있습니다.  
대화가 쌓일수록 메시지 목록이 길어지고, 컨텍스트 창을 초과하거나 응답이 느려집니다.

그 전에 **무엇을 유지하고 무엇을 버릴지** 결정해야 합니다. 전략은 두 가지입니다.

```
전략 1 – Trimming  : 오래된 메시지를 잘라내고 최근 N개만 유지  (빠르지만 정보 손실)
전략 2 – Summarization : 오래된 메시지를 요약으로 압축          (정보 보존, LLM 호출 추가)
```

### 전략 1: Trimming

```
[msg1, msg2, msg3, msg4, msg5]  →  [msg3, msg4, msg5]  (최근 3개만 유지)
```

In [ ]:
llm_base = ChatOpenAI(model=MODEL, temperature=0)

# 긴 대화 시뮬레이션 ("앞에서 말한 정보"를 마지막 질문이 요구하도록 구성)
# 의도: Trimming을 하면 앞부분(중요 정보)이 잘려서 마지막 질문에 제대로 답하기 어려워지는 걸 보여줍니다.
long_conversation = [
    SystemMessage(content="당신은 친절한 어시스턴트입니다."),

    # (중요 정보) 초반에만 등장
    HumanMessage(content="내 비밀코드는 7F-92야. 이건 꼭 기억해."),
    AIMessage(content="알겠어요. 비밀코드는 7F-92로 기억해둘게요."),

    # (잡담/노이즈) 길이를 늘려서 앞 대화가 잘리도록 만듦
    HumanMessage(content="참고로 나는 커피를 좋아해."),
    AIMessage(content="좋아요. 커피 취향도 기억해둘게요."),
    HumanMessage(content="오늘 수업은 에이전트랑 LangGraph지?"),
    AIMessage(content="맞아요. 에이전트와 LangGraph를 다룹니다."),
    HumanMessage(content="그럼 Tool Calling은 뭐야?"),
    AIMessage(content="LLM이 어떤 도구를 어떤 인자로 호출할지 결정하는 인터페이스예요."),

    # (마지막 질문) 앞의 중요한 정보를 정확히 요구
    HumanMessage(content="내 비밀코드가 뭐였지? 코드만 정확히 말해줘."),
]

print(f"원본 메시지 수: {len(long_conversation)}")

# trim_messages: 토큰 기준으로 오래된 메시지를 잘라냅니다
trimmer = trim_messages(
    max_tokens=70,
    strategy="last",            # 최근 메시지 유지
    token_counter=llm_base,
    include_system=True,        # 시스템 메시지는 항상 유지
    allow_partial=False,
    start_on="human",           # human 메시지에서 시작
)

trimmed = trimmer.invoke(long_conversation)
print(f"Trimming 후 메시지 수: {len(trimmed)}")
print("\n유지된 메시지들:")
for msg in trimmed:
    role = type(msg).__name__
    print(f"  [{role:15s}] {msg.content[:60]}")

In [ ]:
# Trimming을 LangGraph Node에 통합하기

class AgentState(TypedDict):
    messages: Annotated[list, add_messages]

trimmer_node = trim_messages(
    max_tokens=100,
    strategy="last",
    token_counter=llm_base,
    include_system=True,
    allow_partial=False,
    start_on="human",
)

def call_llm_with_trim(state: AgentState) -> dict:
    """Trimming 후 LLM 호출"""
    trimmed_messages = trimmer_node.invoke(state["messages"])
    response = llm_base.invoke(trimmed_messages)
    return {"messages": [response]}

# 그래프 조립 (단순 대화 예시)
trim_builder = StateGraph(AgentState)
trim_builder.add_node("call_llm", call_llm_with_trim)
trim_builder.add_edge(START, "call_llm")
trim_builder.add_edge("call_llm", END)
trim_app = trim_builder.compile()

# 테스트: 긴 대화 히스토리 후에 질문해도 최근 메시지만 LLM에 전달됨
result = trim_app.invoke({"messages": long_conversation})
print("===== Trimming 적용 Agent 답변 =====")
print(result["messages"][-1].content)

### 전략 2: Summarization

오래된 대화를 **LLM으로 요약**한 뒤 원본은 버립니다. 정보 손실이 적지만 LLM 호출이 한 번 더 필요합니다.

```
[msg1, msg2, msg3, msg4, msg5]
      ↓ 앞 3개를 요약
[요약: "사용자가 파이썬 질문을 했고 답변받음", msg4, msg5]
```

In [ ]:
# summary 필드를 State에 추가
class SummaryState(TypedDict):
    messages: Annotated[list, add_messages]
    summary: str   # 이전 대화 요약


llm_sum = ChatOpenAI(model=MODEL, temperature=0)


def summarize_if_needed(state: SummaryState) -> dict:
    """메시지가 6개 이상이면 오래된 것들을 요약하고 최근 2개만 유지"""
    messages = state["messages"]

    if len(messages) <= 6:
        return {}   # 변경 없음

    messages_to_summarize = messages[:-2]   # 최근 2개 제외

    summary_response = llm_sum.invoke([
        *messages_to_summarize,
        HumanMessage(content="위 대화를 2~3문장으로 요약해줘.")
    ])
    new_summary = summary_response.content

    print(f"  [요약 실행] {len(messages_to_summarize)}개 → 요약본으로 압축")
    print(f"  요약 내용: {new_summary[:80]}...")

    # messages를 최근 2개로 교체, summary 업데이트
    return {"messages": messages[-2:], "summary": new_summary}


def call_llm_with_summary(state: SummaryState) -> dict:
    """요약이 있으면 시스템 메시지로 주입한 뒤 LLM 호출"""
    messages = state["messages"]
    summary  = state.get("summary", "")

    if summary:
        messages = [SystemMessage(content=f"이전 대화 요약:\n{summary}")] + messages

    response = llm_sum.invoke(messages)
    return {"messages": [response]}


# 그래프 조립
sum_builder = StateGraph(SummaryState)
sum_builder.add_node("summarize", summarize_if_needed)
sum_builder.add_node("call_llm",  call_llm_with_summary)
sum_builder.add_edge(START,       "summarize")
sum_builder.add_edge("summarize", "call_llm")
sum_builder.add_edge("call_llm",  END)

sum_app = sum_builder.compile()
print("✅ Summarization Agent 구성 완료")

In [ ]:
# 테스트: 6개 이상의 대화 후 "앞에서 말한 정보"를 묻는 질문
# 의도: Summarization을 하면 앞 대화가 요약(summary)으로 남아서, 마지막 질문에도 정확히 답할 수 있음을 보여줍니다.

test_messages = [
    SystemMessage(content="당신은 친절한 어시스턴트입니다."),

    # (중요 정보) 초반에만 등장 → 요약에 포함되어야 함
    HumanMessage(content="내 비밀코드는 7F-92야. 이건 꼭 기억해."),
    AIMessage(content="알겠어요. 비밀코드는 7F-92로 기억해둘게요."),

    # (잡담/노이즈) 길이를 늘려서 요약이 실행되도록 함
    HumanMessage(content="참고로 나는 커피를 좋아해."),
    AIMessage(content="좋아요. 커피 취향도 기억해둘게요."),
    HumanMessage(content="오늘 수업은 에이전트랑 LangGraph지?"),
    AIMessage(content="맞아요. 에이전트와 LangGraph를 다룹니다."),

    # (마지막 질문) 앞에서 준 정보를 정확히 요구
    HumanMessage(content="내 비밀코드가 뭐였지? 코드만 정확히 말해줘."),
]

print(f"입력 메시지 수: {len(test_messages)} → 요약 실행 예상")
print()

result = sum_app.invoke({"messages": test_messages, "summary": ""})
print("\n===== 최종 답변 =====")
print(result["messages"][-1].content)

### 어떤 전략을 언제 쓸까?

| 상황 | 추천 전략 |
|:--- |:--- |
| 단순 Q&A, 짧은 대화 | Trimming |
| 긴 대화, 맥락이 중요한 경우 | Summarization |
| Tool 결과물이 많고 긴 경우 | 중요도 기반 필터링 |

실제 서비스에서는 두 전략을 조합합니다.  
"최근 N개는 그대로 유지하되, 그 이전은 요약"처럼 쓰는 게 일반적입니다.


## 💾 3. 메모리 – Checkpointer

### Short-term vs Long-term Memory

| | Short-term Memory | Long-term Memory |
|:--- |:--- |:--- |
| **범위** | 같은 대화 세션 안 | 대화 종료 후에도 유지 |
| **LangGraph** | State (기본 제공) | Checkpointer + 외부 DB |
| **용도** | 현재 대화 맥락 유지 | 사용자 이름, 선호도 등 |
| **소멸 시점** | 대화가 끝나면 사라짐 | 명시적으로 삭제하지 않는 한 유지 |

```
대화 1: "내 이름은 Alice야"
대화 1 종료 → short-term memory 소멸

대화 2: "내 이름이 뭐였지?"
→ short-term만 있으면 모른다
→ long-term에 저장해뒀다면 안다
```

LangGraph의 **Checkpointer**는 `thread_id` 기반으로 State를 저장합니다.  
같은 `thread_id`로 실행하면 이전 State가 복원되어 대화가 이어집니다.

In [ ]:
# Checkpointer가 포함된 Agent 구성

memory = MemorySaver()   # 개발용 메모리 내 저장소
                         # 실제 서비스: SqliteSaver, PostgresSaver 등으로 교체

llm_mem = ChatOpenAI(model=MODEL, temperature=0)

def call_llm_mem(state: AgentState) -> dict:
    response = llm_mem.invoke(state["messages"])
    return {"messages": [response]}

mem_builder = StateGraph(AgentState)
mem_builder.add_node("call_llm", call_llm_mem)
mem_builder.add_edge(START, "call_llm")
mem_builder.add_edge("call_llm", END)

# checkpointer=memory 를 추가하면 각 Step마다 State가 저장됩니다
mem_app = mem_builder.compile(checkpointer=memory)

print("✅ Checkpointer 포함 Agent 구성 완료")

In [ ]:
# thread_id로 대화 세션을 구분합니다
# 같은 thread_id → 이전 State 복원 → 대화 지속
# 다른 thread_id → 완전히 새로운 대화

config_alice = {"configurable": {"thread_id": "user-alice-001"}}

print("===== Alice의 첫 번째 대화 =====")
result1 = mem_app.invoke(
    {"messages": [HumanMessage(content="내 이름은 Alice야. 안녕!")]},
    config=config_alice
)
print(result1["messages"][-1].content)

print("\n===== Alice의 두 번째 대화 (같은 thread_id) =====")
result2 = mem_app.invoke(
    {"messages": [HumanMessage(content="내 이름이 뭐라고 했지?")]},
    config=config_alice
)
print(result2["messages"][-1].content)

In [ ]:
# 다른 thread_id는 이전 대화 기록이 없는 완전히 새로운 세션
config_bob = {"configurable": {"thread_id": "user-bob-002"}}

print("===== Bob의 대화 (다른 thread_id) =====")
result3 = mem_app.invoke(
    {"messages": [HumanMessage(content="내 이름이 뭐라고 했지?")]},
    config=config_bob
)
print(result3["messages"][-1].content)
print("\n→ Bob은 Alice의 대화 기록이 없으므로 이름을 모릅니다.")

In [ ]:
# 저장된 State 조회: 특정 시점의 대화 기록을 꺼내볼 수 있습니다
state_snapshot = mem_app.get_state(config_alice)

print("===== Alice의 저장된 State 조회 =====")
print(f"총 메시지 수: {len(state_snapshot.values['messages'])}")
print("\n메시지 목록:")
for msg in state_snapshot.values["messages"]:
    role = type(msg).__name__
    print(f"  [{role:13s}] {msg.content[:60]}")

> **이게 LangGraph를 단순한 루프 코드와 구별 짓는 핵심 기능입니다.**  
> Checkpointer 덕분에:
> - **대화 지속성**: 같은 `thread_id`로 다시 실행하면 이전 대화가 복원됩니다.
> - **디버깅**: 특정 시점의 State를 꺼내서 확인하거나, 그 시점부터 다시 실행할 수 있습니다.
> - **Human-in-the-Loop**: 특정 지점에서 멈추고, 사람의 응답을 받은 뒤 재개할 수 있습니다.

---
## 📐 4. 구조화된 응답 (Structured Output)

### 자유 텍스트의 한계

LLM은 기본적으로 자유 텍스트를 반환합니다.  
애플리케이션에서 특정 값을 꺼내려면 불안정한 텍스트 파싱이 필요합니다.

```
"제품 이름은 맥북이고, 가격은 150만원이며, 재고는 있습니다."
→ price 값만 꺼내려면? 텍스트 파싱 필요 → 불안정!
```

구조화된 응답은 LLM이 **처음부터 정해진 스키마에 맞는 데이터를 반환**하도록 강제합니다.

In [ ]:
# Pydantic 모델로 스키마 정의
class ProductInfo(BaseModel):
    name:     str  = Field(description="제품 이름")
    price:    int  = Field(description="가격 (원 단위 숫자만, 예: 1500000)")
    in_stock: bool = Field(description="재고 여부")
    category: str  = Field(description="제품 카테고리 (예: 전자제품, 식품 등)")


llm_struct = ChatOpenAI(model=MODEL, temperature=0)

# with_structured_output()으로 LLM이 스키마에 맞게 반환하도록 강제
# 내부적으로 Function Calling 또는 JSON 모드를 활용합니다
structured_llm = llm_struct.with_structured_output(ProductInfo)

result = structured_llm.invoke(
    "맥북 프로 M3는 약 250만원이고 현재 재고가 있는 노트북 제품입니다."
)

print("===== 구조화된 응답 결과 =====")
print(f"제품명  : {result.name}")
print(f"가격    : {result.price:,}원")
print(f"재고    : {'있음' if result.in_stock else '없음'}")
print(f"카테고리: {result.category}")
print(f"\n타입 확인: {type(result)}  ← dict가 아니라 Pydantic 객체")

### Agent 흐름 제어에도 활용

구조화된 응답은 단순히 데이터 추출에만 쓰이는 게 아닙니다.  
Agent가 **다음에 무엇을 할지 결정**하는 데도 활용할 수 있습니다.  
이렇게 하면 Conditional Edge의 분기 조건을 LLM의 추론에 맡기면서도, 그 결과를 **안정적으로 코드에서 다룰 수 있습니다.**

In [ ]:
class NextAction(BaseModel):
    action: Literal["search", "calculate", "answer"] = Field(
        description="다음 행동: search(정보 검색 필요), calculate(계산 필요), answer(바로 답변 가능)"
    )
    reason: str = Field(description="이 행동을 선택한 이유")
    query:  str = Field(description="검색이나 계산에 사용할 구체적인 쿼리")


action_llm = ChatOpenAI(model=MODEL, temperature=0).with_structured_output(NextAction)

test_questions = [
    "현재 달러 환율이 얼마야?",
    "2의 10제곱은 얼마야?",
    "파이썬 언어가 처음 만들어진 연도는?",
]

for q in test_questions:
    decision = action_llm.invoke(f"질문: {q}")
    print(f"\n질문  : {q}")
    print(f"  행동 : {decision.action}")
    print(f"  이유 : {decision.reason}")
    print(f"  쿼리 : {decision.query}")

In [ ]:
# Structured Output을 Conditional Edge에 적용한 Agent 예시

class RouterState(TypedDict):
    messages:    Annotated[list, add_messages]
    next_action: str


llm_router = ChatOpenAI(model=MODEL, temperature=0)
action_router = llm_router.with_structured_output(NextAction)


def router_node(state: RouterState) -> dict:
    """LLM이 다음 행동을 구조화된 형식으로 결정"""
    question = state["messages"][-1].content
    decision = action_router.invoke(f"질문: {question}")
    print(f"  [Router] action={decision.action}, query={decision.query}")
    return {"next_action": decision.action}


def search_node(state: RouterState) -> dict:
    return {"messages": [AIMessage(content="(검색 결과를 가져와서 답변했습니다.)")]}


def calculate_node(state: RouterState) -> dict:
    return {"messages": [AIMessage(content="(계산 결과를 반환했습니다.)")]}


def direct_answer_node(state: RouterState) -> dict:
    resp = llm_router.invoke(state["messages"])
    return {"messages": [resp]}


def route_by_action(state: RouterState) -> str:
    return state["next_action"]   # "search" | "calculate" | "answer"


router_builder = StateGraph(RouterState)
router_builder.add_node("router",    router_node)
router_builder.add_node("search",    search_node)
router_builder.add_node("calculate", calculate_node)
router_builder.add_node("answer",    direct_answer_node)

router_builder.add_edge(START, "router")
router_builder.add_conditional_edges(
    "router",
    route_by_action,
    {"search": "search", "calculate": "calculate", "answer": "answer"}
)
router_builder.add_edge("search",    END)
router_builder.add_edge("calculate", END)
router_builder.add_edge("answer",    END)

router_app = router_builder.compile()

print("✅ Structured Output 기반 Router Agent 구성 완료")

# 그래프 시각화
try:
    from IPython.display import Image, display
    display(Image(router_app.get_graph().draw_mermaid_png()))
except Exception:
    print(router_app.get_graph().draw_mermaid())

In [ ]:
# 테스트
for q in test_questions:
    print(f"\n질문: {q}")
    res = router_app.invoke({"messages": [HumanMessage(content=q)], "next_action": ""})
    print(f"답변: {res['messages'][-1].content[:80]}")

> **주의할 점**: 모든 모델이 구조화된 응답을 동일하게 잘 지원하지는 않습니다.  
> Pydantic의 `Field(description=...)` 에 **명확한 설명**을 쓰는 것이 중요합니다.  
> LLM이 이 설명을 보고 어떤 값을 채워야 할지 판단하기 때문입니다.

---
## 🙋 5. Human-in-the-Loop

### 개념

Agent가 모든 결정을 자율적으로 내리는 게 항상 좋은 건 아닙니다.  
이메일 발송, DB 수정, 결제 같은 **중요한 행동**은 사람이 확인한 후 실행해야 합니다.

```
Agent 실행 중...
→ "이메일을 발송할 예정입니다. 승인하시겠습니까?"  (일시 정지)
→ 사람: "승인" / "거부" / "내용 수정"
→ Agent 재개
```

### LangGraph에서의 구현: `interrupt()`

Node 안에서 `interrupt()`를 호출하면 그래프 실행이 그 지점에서 **멈춥니다**.  
Checkpointer가 현재 State를 저장하고, 사람이 응답하면 그 State를 복원해서 실행을 이어갑니다.

```python
from langgraph.types import interrupt

def approval_node(state):
    human_input = interrupt({"message": "이 작업을 실행할까요?"})
    # ↑ 이 지점에서 실행이 멈추고 사람의 입력을 기다립니다
    ...
```

In [ ]:
# HITL State 정의
class HITLState(TypedDict):
    messages:       Annotated[list, add_messages]
    planned_action: str
    status:         str


llm_hitl = ChatOpenAI(model=MODEL, temperature=0)


def plan_action(state: HITLState) -> dict:
    """수행할 작업을 계획하는 Node"""
    user_request = state["messages"][-1].content
    planned = f"'{user_request}'에 대한 공지 이메일을 all-users@company.com 으로 발송"
    print(f"  [계획] {planned}")
    return {"planned_action": planned}


def approval_node(state: HITLState) -> dict:
    """사람의 승인을 기다리는 Node — interrupt()가 핵심"""
    planned = state["planned_action"]

    # interrupt() 호출 → 실행 일시 정지 + State 저장
    human_response = interrupt({
        "message": "다음 작업을 실행할까요?",
        "planned_action": planned,
        "options": ["승인", "거부"]
    })

    return {"status": "approved" if human_response == "승인" else "rejected"}


def execute_action(state: HITLState) -> dict:
    """승인 여부에 따라 작업을 실행하거나 취소"""
    if state["status"] == "approved":
        msg = f"✅ 작업 완료: {state['planned_action']}"
    else:
        msg = f"❌ 작업이 거부되었습니다."
    return {"messages": [AIMessage(content=msg)]}


# Checkpointer 필수: interrupt()가 State를 저장하고 복원하는 데 필요합니다
hitl_memory = MemorySaver()

hitl_builder = StateGraph(HITLState)
hitl_builder.add_node("plan",     plan_action)
hitl_builder.add_node("approval", approval_node)
hitl_builder.add_node("execute",  execute_action)
hitl_builder.add_edge(START,      "plan")
hitl_builder.add_edge("plan",     "approval")
hitl_builder.add_edge("approval", "execute")
hitl_builder.add_edge("execute",  END)

# interrupt_before=["approval"]: approval 노드 실행 직전에 멈춥니다
hitl_agent = hitl_builder.compile(
    checkpointer=hitl_memory,
    interrupt_before=["approval"]
)

print("✅ Human-in-the-Loop Agent 구성 완료")

# 그래프 시각화
try:
    from IPython.display import Image, display
    display(Image(hitl_agent.get_graph().draw_mermaid_png()))
except Exception:
    print(hitl_agent.get_graph().draw_mermaid())

In [ ]:
# ── 시나리오 A: 승인 ──────────────────────────────────────────
config_a = {"configurable": {"thread_id": "hitl-approve-001"}}

initial_input = {
    "messages": [HumanMessage(content="신규 서비스 출시 공지")],
    "planned_action": "",
    "status": ""
}

print("===== 1단계: Agent 실행 → approval 직전에 중단됨 =====")
for event in hitl_agent.stream(initial_input, config=config_a):
    node_name = list(event.keys())[0]
    print(f"  실행된 노드: {node_name}")

# 현재 중단 지점 확인
current_state = hitl_agent.get_state(config_a)
print(f"\n계획된 작업: {current_state.values.get('planned_action')}")
print(f"다음 실행 노드: {current_state.next}")
print("\n⏸️  사람의 승인 대기 중...")

In [ ]:
print("===== 2단계: 사람이 '승인' 결정 → Agent 재개 =====")

# Command(resume=값)으로 interrupt에 응답
for event in hitl_agent.stream(Command(resume="승인"), config=config_a):
    node_name = list(event.keys())[0]
    node_data = event[node_name]
    if "messages" in node_data:
        print(f"\n[{node_name}] → {node_data['messages'][-1].content}")

In [ ]:
# ── 시나리오 B: 거부 ──────────────────────────────────────────
config_b = {"configurable": {"thread_id": "hitl-reject-001"}}

print("===== 거부 시나리오 =====")
# 1단계: 실행 후 중단
for event in hitl_agent.stream(initial_input, config=config_b):
    pass

# 2단계: 거부 응답
for event in hitl_agent.stream(Command(resume="거부"), config=config_b):
    node_name = list(event.keys())[0]
    node_data = event[node_name]
    if "messages" in node_data:
        print(f"[{node_name}] → {node_data['messages'][-1].content}")

> **왜 중요한가?**  
> 단순히 "사람이 확인하는" 기능이 아닙니다.  
> 이 패턴이 있어야 **신뢰할 수 있는 자동화**가 가능합니다.  
>
> | | 완전 자동화 | Human-in-the-Loop |
> |:--- |:--- |:--- |
> | 장점 | 빠름, 사람 개입 불필요 | 중요한 결정은 사람이 검토 |
> | 단점 | 실수해도 되돌리기 어려움 | 사람의 확인 시간 필요 |
> | 적합한 상황 | 위험도 낮은 반복 작업 | 이메일 발송, 결제, DB 수정 등 |
>
> Human-in-the-Loop은 Agent의 **자율성 경계를 조절하는 핵심 도구**입니다.
---
## 📋 전체 정리

| 주제 | 핵심 질문 | 핵심 도구 |
|:--- |:--- |:--- |
| **컨텍스트 관리** | 무엇을 LLM에 전달할 것인가 | `trim_messages`, Summarization Node |
| **메모리** | 대화를 얼마나 오래 기억할 것인가 | `MemorySaver`, `thread_id` |
| **구조화된 응답** | LLM 출력을 코드에서 안정적으로 다루려면 | `with_structured_output`, Pydantic |
| **Human-in-the-Loop** | 어디서 사람이 개입해야 하는가 | `interrupt()`, `Command(resume=...)` |

```
LangGraph는 이 구조를 통해
→ 투명하게 동작하는 Agent (stream으로 실시간 추적)
→ 기억하는 Agent (Checkpointer + thread_id)
→ 안전한 Agent (Human-in-the-Loop)
→ 코드와 통합되는 Agent (Structured Output)
를 만들 수 있게 해줍니다.
```

